# Student Performance Prediction System

## Project Objective
Build and evaluate machine-learning regression models that estimate a student's exam score from study habits, attendance, previous scores, and other student-related factors.

**Workflow:** Data loading → understanding → cleaning → exploratory analysis → preprocessing → model training → evaluation → prediction → model saving.

> This is an educational prediction project. Predictions are estimates, not guaranteed exam results.


In [ ]:
import pandas as pd
import numpy as np


## Step 1 — Load the Dataset

Load the Kaggle Student Performance Factors dataset into a pandas DataFrame.


In [ ]:
df = pd.read_csv('StudentPerformanceFactors.csv')
print('Dataset Loaded Successfully!')


## Step 2 — Understand the Dataset

Inspect the dataset's dimensions, sample records, column names, data types, and descriptive statistics before making changes.


In [ ]:
# Step 2: Check dataset dimensions
print("Dataset Shape:", df.shape)

# Step 3: Display first 5 rows
display(df.head())

# Step 4: Check column names
print("\nColumn Names:")
print(df.columns.tolist())

# Step 5: Check data types and non-null values
print("\nDataset Information:")
df.info()

# Step 6: Generate statistical summary
print("\nStatistical Summary:")
display(df.describe(include="all").T)


## Step 3 — Data Cleaning and Missing Values

Check missing values, missing-value percentages, duplicate rows, and categorical values.


In [ ]:
# 1. Count missing values in each column
print("Missing Values Per Column:")
print(df.isnull().sum())

# 2. Calculate total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

# 3. Calculate missing-value percentage
print("\nMissing Value Percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

# 4. Count duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# 5. Check unique values in categorical columns
categorical_columns = df.select_dtypes(include="object").columns

print("\nUnique Values in Categorical Columns:")
for column in categorical_columns:
    print(f"{column}: {df[column].unique()}")


In [ ]:
# Step 3: Handle missing categorical values

missing_columns = [
    "Teacher_Quality",
    "Parental_Education_Level",
    "Distance_from_Home"
]

# Check the most frequent category before filling
for column in missing_columns:
    print(f"{column} mode:", df[column].mode()[0])

# Fill missing values using each column's mode
for column in missing_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

# Verify the cleaning
print("\nMissing values after cleaning:")
print(df[missing_columns].isnull().sum())

print("\nTotal missing values remaining:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


## Step 4 — Exploratory Data Analysis (EDA)

Use summary statistics and visualizations to understand the distribution of exam scores and explore how study hours, attendance, and numerical features relate to exam performance.


### 4.1 Numerical Summary

Review numerical summary statistics and plot the distribution of `Exam_Score`.


In [ ]:
# Step 4.1: Statistical analysis of numerical columns

numerical_columns = df.select_dtypes(include="number").columns

display(df[numerical_columns].describe().T)


### 4.2 Distribution of Exam Scores


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.hist(df["Exam_Score"], bins=20, edgecolor="black")

plt.title("Distribution of Student Exam Scores")
plt.xlabel("Exam Score")
plt.ylabel("Number of Students")
plt.grid(axis="y", alpha=0.3)

plt.show()


### 4.3 Study Hours vs. Exam Score


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Hours_Studied"],
    df["Exam_Score"],
    alpha=0.5
)

plt.title("Study Hours vs Exam Score")
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.grid(alpha=0.3)

plt.show()


### 4.4 Attendance vs. Exam Score

Inspect the relationship between attendance and exam score. A visible association does not by itself establish causation.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Attendance"],
    df["Exam_Score"],
    alpha=0.4
)

plt.title("Attendance vs Exam Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Exam Score")
plt.grid(alpha=0.3)

plt.show()


### 4.5 Correlation Analysis

Calculate correlations among numerical columns and inspect how each numerical feature is correlated with `Exam_Score`. Correlation measures association, not causation.


In [ ]:
import seaborn as sns

correlation_matrix = df.select_dtypes(include="number").corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()

print("\nCorrelation with Exam Score:")
print(
    correlation_matrix["Exam_Score"]
    .sort_values(ascending=False)
)


## Step 5 — Investigate Potential Outliers

Inspect values outside the expected exam-score range and unusually high study-hour values. Investigate these records before deciding whether any correction is justified.


In [ ]:
# Check exam scores outside the expected 0–100 range
print("Minimum Exam Score:", df["Exam_Score"].min())
print("Maximum Exam Score:", df["Exam_Score"].max())

# Inspect records with scores above 100
high_scores = df[df["Exam_Score"] > 100]

print("\nRecords with Exam_Score above 100:")
display(high_scores)

print("Number of records above 100:", len(high_scores))

# Check unusually high study hours
print("\nRecords with Hours_Studied above 40:")
display(df[df["Hours_Studied"] > 40])


## Step 6 — Prepare a Separate Modeling Dataset

Keep the original cleaned `df` unchanged and create `model_df` for modeling. Inspect the target variable and identify scores outside the expected 0–100 range.


In [ ]:
# Step 8.1: Create a separate modeling copy
model_df = df.copy()

# Check the target variable
print("Target column: Exam_Score")
print("Target data type:", model_df["Exam_Score"].dtype)

print("\nTarget summary:")
print(model_df["Exam_Score"].describe())

# Count scores outside the expected 0–100 range
invalid_scores = model_df[
    (model_df["Exam_Score"] < 0) |
    (model_df["Exam_Score"] > 100)
]

print("\nScores outside 0–100:", len(invalid_scores))
display(invalid_scores)


### Step 7 — Handle the Exam Score Above 100

The dataset contains one exam score above 100. For this experiment, the modeling copy caps scores above 100 at 100. This is an assumption for the project—not a verified correction from the dataset creator—so document it when presenting results.


In [ ]:
# Work on the separate modeling copy
model_df = df.copy()

# Count scores above 100 before correction
before = (model_df["Exam_Score"] > 100).sum()
print("Scores above 100 before correction:", before)

# Correct the invalid score to 100
model_df.loc[model_df["Exam_Score"] > 100, "Exam_Score"] = 100

# Verify the correction
print("Maximum Exam Score after correction:", model_df["Exam_Score"].max())
print("Scores above 100 after correction:",
      (model_df["Exam_Score"] > 100).sum())
print("Dataset shape:", model_df.shape)


## Step 8 — Feature Selection and Train-Test Split

Separate the target (`Exam_Score`) from the input features, then split the data into training and testing sets. The test set is held out for evaluation.


In [ ]:
# Step 9.1: Separate input features and target

X = model_df.drop(columns=["Exam_Score"])
y = model_df["Exam_Score"]

print("Input features shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumerical features:")
print(X.select_dtypes(include="number").columns.tolist())

print("\nCategorical features:")
print(X.select_dtypes(include="object").columns.tolist())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


## Step 9 — Preprocessing and Model Training

Build preprocessing into a scikit-learn pipeline so the same transformations are applied consistently during training and prediction.


### Step 9.1 — Build the Preprocessing Pipeline

- Numerical features: fill missing values with the median.
- Categorical features: fill missing values with the most frequent category, then apply one-hot encoding.
- `handle_unknown="ignore"` allows the pipeline to process categories not seen during training.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Identify feature types
numerical_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Preprocessing pipeline created successfully.")


### Step 9.2 — Train a Random Forest Regressor

Train a Random Forest regression pipeline as the first model to establish a baseline.


In [ ]:
# Create the complete ML pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Random Forest model trained successfully!")


### Step 9.3 — Evaluate the Random Forest Model

Evaluate predictions on the held-out test set using MAE, RMSE, and R².


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict exam scores
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Results")
print("------------------------")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")


## Step 10 — Random Forest Baseline Results

Interpret the metrics produced by the evaluation cell above:

- **MAE:** the average absolute difference between actual and predicted exam scores; lower is better.
- **RMSE:** penalizes larger errors more strongly; lower is better.
- **R²:** measures the proportion of target variation explained by the model on this test set; closer to 1 is generally better.

Use the actual printed results from your current notebook run rather than treating previously recorded values as permanent.


## Step 11 — Random Forest Feature Importance

Inspect the Random Forest's feature-importance scores and plot the 15 highest-ranked encoded features. These scores describe how the fitted model uses features; they do not prove that a feature causes higher exam scores.


In [ ]:
# Get the fitted preprocessing components
fitted_preprocessor = model.named_steps["preprocessor"]

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Get Random Forest feature importances
importances = model.named_steps["regressor"].feature_importances_

# Create a sorted feature-importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Top 15 Most Important Encoded Features:")
display(importance_df.head(15))


In [ ]:
top_features = importance_df.head(15).sort_values(
    by="Importance",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## Step 12 — Compare Linear Regression and Random Forest

Train Linear Regression with the same preprocessing and compare both models using the same test set and evaluation metrics. Select the model based on the comparison table rather than assuming one algorithm is always better.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Train Linear Regression using the same preprocessing
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

# Generate predictions
linear_pred = linear_model.predict(X_test)
rf_pred = model.predict(X_test)

# Compare both models
results = []

for name, predictions in [
    ("Linear Regression", linear_pred),
    ("Random Forest", rf_pred)
]:
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2 Score": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))


## Step 13 — Compare Actual and Predicted Scores

Display actual scores, predictions from the selected model, and absolute errors for a small sample of test records. These rows illustrate predictions; the full test-set metrics remain the main evaluation.


In [ ]:
# Use the Linear Regression model selected from the comparison
best_model = linear_model

# Compare actual and predicted exam scores
comparison_df = pd.DataFrame({
    "Actual_Score": y_test.iloc[:10].values,
    "Predicted_Score": best_model.predict(X_test.iloc[:10]).round(2)
})

comparison_df["Absolute_Error"] = (
    comparison_df["Actual_Score"] -
    comparison_df["Predicted_Score"]
).abs().round(2)

display(comparison_df)


## Step 14 — Predict for a Sample New Student

Create one example student using the same input columns expected by the trained pipeline, then generate an estimated exam score.


In [ ]:
# Create one sample student using valid dataset categories
sample_student = pd.DataFrame([{
    "Hours_Studied": 25,
    "Attendance": 90,
    "Parental_Involvement": "High",
    "Access_to_Resources": "High",
    "Extracurricular_Activities": "Yes",
    "Sleep_Hours": 7,
    "Previous_Scores": 80,
    "Motivation_Level": "High",
    "Internet_Access": "Yes",
    "Tutoring_Sessions": 2,
    "Family_Income": "Medium",
    "Teacher_Quality": "High",
    "School_Type": "Private",
    "Peer_Influence": "Positive",
    "Physical_Activity": 3,
    "Learning_Disabilities": "No",
    "Parental_Education_Level": "College",
    "Distance_from_Home": "Near",
    "Gender": "Male"
}])

predicted_score = best_model.predict(sample_student)[0]

print(f"Predicted Exam Score: {predicted_score:.2f}")


## Step 15 — Save the Selected Model

Save the complete preprocessing-and-regression pipeline so future predictions use the same transformations as training.


In [ ]:
import joblib

# Save the complete preprocessing + model pipeline
joblib.dump(best_model, "student_performance_model.pkl")

print("Model saved successfully!")


## Step 16 — Reload and Verify the Saved Model

Load the saved pipeline and predict for the same sample student. The result should match the prediction from the original pipeline, apart from negligible numerical differences.


In [ ]:
# Load the saved model
loaded_model = joblib.load("student_performance_model.pkl")

# Predict using the loaded model
loaded_prediction = loaded_model.predict(sample_student)[0]

print(f"Prediction from loaded model: {loaded_prediction:.2f}")


## Step 17 — Save Project Artifacts

Export the cleaned modeling dataset, model-comparison table, and sample-prediction table as CSV files. The trained model is saved separately as a `.pkl` file.


In [ ]:
import pandas as pd

# Save cleaned dataset after the score correction
model_df.to_csv("student_performance_cleaned.csv", index=False)

# Save model comparison results
results_df.to_csv("model_comparison.csv", index=False)

# Save sample predictions
comparison_df.to_csv("sample_predictions.csv", index=False)

print("Project artifacts saved successfully!")
print("- student_performance_cleaned.csv")
print("- model_comparison.csv")
print("- sample_predictions.csv")
print("- student_performance_model.pkl")


## Step 18 — Download Project Files from Google Colab

Run this cell to download the exported files to your computer. If Colab asks for permission, allow the downloads.


In [ ]:
from google.colab import files

files.download("student_performance_cleaned.csv")
files.download("model_comparison.csv")
files.download("sample_predictions.csv")
files.download("student_performance_model.pkl")


## Step 19 — Limitations and Responsible Interpretation

- The model predicts an estimated score from patterns in this dataset; it cannot guarantee an individual student's result.
- The score-above-100 adjustment is an explicit modeling assumption and should be documented.
- Feature importance and correlation show associations, not cause-and-effect relationships.
- The current metrics come from one train-test split. Cross-validation and checks on other data would provide stronger evidence about generalization.


## Step 20 — Project Conclusion and Final Summary

This section summarizes the selected model and its measured performance from the current train-test split. Run it after all previous cells, including model comparison and artifact saving.

**How to read the result:** MAE and RMSE are measured in exam-score points. R² is a unitless score. These results describe performance on this particular test split; use cross-validation and further checks before making strong claims about generalization.


In [ ]:
# Select the model with the lowest MAE in the comparison table
best_result = results_df.sort_values(by="MAE", ascending=True).iloc[0]

print("=" * 60)
print("STUDENT PERFORMANCE PREDICTION SYSTEM — FINAL SUMMARY")
print("=" * 60)
print(f"Dataset rows used: {len(model_df):,}")
print(f"Input features: {X.shape[1]}")
print(f"Training records: {len(X_train):,}")
print(f"Testing records: {len(X_test):,}")
print()
print(f"Best model by test MAE: {best_result['Model']}")
print(f"MAE: {best_result['MAE']:.4f} score points")
print(f"RMSE: {best_result['RMSE']:.4f} score points")
print(f"R²: {best_result['R2 Score']:.4f}")
print()
print("Saved artifacts:")
print("- student_performance_model.pkl")
print("- student_performance_cleaned.csv")
print("- model_comparison.csv")
print("- sample_predictions.csv")
print()
print("Note: These metrics are based on one held-out test split.")
print("They are not a guarantee of future prediction accuracy.")
